Lien Google doc sur les modèles de diffusion

https://docs.google.com/document/d/1KRHOdVA75olgOQNJOSlI29OYkfh-zTTXwaEssPlZ8rA/edit?usp=sharing

In [ ]:
# Google Drive
from google.colab import drive
drive.mount('/content/drive')


# Installations

In [ ]:
# Installer les dépendances nécessaires
!pip install -U peft transformers diffusers accelerate

# Cloner le repo diffusers (Hugging Face)
!git clone https://github.com/huggingface/diffusers
%cd diffusers/examples/dreambooth

# Installer les requirements spécifiques à SDXL
!pip install -U -r requirements_sdxl.txt

# Installer la dernière version de diffusers depuis GitHub
!pip uninstall -y diffusers
!pip install git+https://github.com/huggingface/diffusers.git


  Using cached peft-0.15.2-py3-none-any.whl.metadata (13 kB)
Using cached peft-0.15.2-py3-none-any.whl (411 kB)
  Attempting uninstall: peft
    Found existing installation: peft 0.15.1
    Uninstalling peft-0.15.1:
      Successfully uninstalled peft-0.15.1


In [ ]:
#@title Login to HuggingFace 🤗

#@markdown You need to accept the model license before downloading or using the Stable Diffusion weights. Please, visit the [model card](https://huggingface.co/stabilityai/stable-diffusion-2), read the license and tick the checkbox if you agree. You have to be a registered user in 🤗 Hugging Face Hub, and you'll also need to use an access token for the code to work.
# https://huggingface.co/settings/tokens
!mkdir -p ~/.huggingface
HUGGINGFACE_TOKEN = "" #@param {type:"string"}
!echo -n "{HUGGINGFACE_TOKEN}" > ~/.huggingface/token

In [ ]:
# Configuration par défaut de Accelerate
!accelerate config default

# Forcer une version stable de `peft`
!pip install peft==0.15.1


# Fine Tuning

In [ ]:
# Lancer le fine-tuning avec DreamBooth LoRA pour SDXL
!accelerate launch train_dreambooth_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --instance_data_dir="/content/drive/MyDrive/Dreambooth/Driss" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --output_dir="/content/drive/MyDrive/Dreambooth/lora-xl-driss-best" \
  --instance_prompt="a photo of pss person" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=2 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=1000 \
  --rank=16 \
  --checkpointing_steps=100 \
  --train_text_encoder \
  --seed="42" \
  --mixed_precision="fp16" \
  --gradient_checkpointing


In [ ]:
# Génération d'image avec le modèle fine-tuné

from diffusers import DiffusionPipeline
import torch

# Charger le pipeline SDXL
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)

# Optimisation mémoire
pipe.enable_model_cpu_offload()

# Charger les poids LoRA fine-tunés
pipe.load_lora_weights(
    "/content/drive/MyDrive/Dreambooth/lora-xl-driss-best",
    weight_name="pytorch_lora_weights.safetensors",
    local_files_only=True
)


# Génération de l'image

In [ ]:
# Générer et afficher une image personnalisée
prompt = "a realistic photo of pss person standing in Paris, with the Eiffel Tower in the background"
image = pipe(
    prompt=prompt,
    num_inference_steps=200,
    height=1024,
    width=1024,
    guidance_scale=7.5
).images[0]

# Sauvegarder et afficher
image.save("/content/drive/MyDrive/Dreambooth/test_driss_1.png")
image.show()
image
